This script generates the following pan-India rasters:
1. Distance to roads, based on the PMGSY II Road Network
2. Distance to water sources, based on in-lab LULC

# Set up

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-indiasat')

# Config

In [ ]:
err = ee.ErrorMargin(100)
scale = 100
year = 2022

india = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'India'))
kolar = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2').filter(ee.Filter.eq("ADM2_NAME","Kolar"))
karnataka = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', "Karnataka")).first()

indiaDistricts = ee.FeatureCollection("projects/ee-indiasat/assets/india_district_boundaries")
indiaACZs = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
india = indiaACZs

# Utils

In [ ]:
# State names and base asset path
# Goa and J&K are missing
states = [
    'andhra_pradesh', 'arunachal_pradesh', 'assam', 'bihar', 'chhattisgarh',
    'gujarat', 'haryana', 'himachal_pradesh', 'jharkhand',
    'karnataka', 'kerala', 'madhya_pradesh', 'maharashtra', 'manipur',
    'meghalaya', 'mizoram', 'nagaland', 'odisha', 'punjab', 'rajasthan',
    'sikkim', 'tamil_nadu', 'telangana', 'tripura', 'uttar_pradesh',
    'uttarakhand', 'west_bengal', 'jammu_and_kashmir', 'ladakh'
]

gaulNames = {
    'andhra_pradesh': 'Andhra Pradesh',
    'arunachal_pradesh': 'Arunachal Pradesh',
    'assam': 'Assam',
    'bihar': 'Bihar',
    'chhattisgarh': 'Chhattisgarh',
    'gujarat': 'Gujarat',
    'haryana': 'Haryana',
    'himachal_pradesh': 'Himachal Pradesh',
    'jharkhand': 'Jharkhand',
    'karnataka': 'Karnataka',
    'kerala': 'Kerala',
    'madhya_pradesh': 'Madhya Pradesh',
    'maharashtra': 'Maharashtra',
    'manipur': 'Manipur',
    'meghalaya': 'Meghalaya',
    'mizoram': 'Mizoram',
    'nagaland': 'Nagaland',
    'odisha': 'Orissa',
    'punjab': 'Punjab',
    'rajasthan': 'Rajasthan',
    'sikkim': 'Sikkim',
    'tamil_nadu': 'Tamil Nadu',
    'telangana': 'Andhra Pradesh', # GAUL 2015 is not updated to reflect Telangana
    'tripura': 'Tripura',
    'uttar_pradesh': 'Uttar Pradesh',
    'uttarakhand': 'Uttarakhand',
    'west_bengal': 'West Bengal'
}

asset_base_path = 'projects/ee-corestackdev/assets/datasets/Road_DRRP/'
india = ee.FeatureCollection('FAO/GAUL/2015/level0') \
           .filter(ee.Filter.eq('ADM0_NAME', 'India')) \
           .geometry()

scale = 100

In [ ]:
def getStateRoi(state):
  gaulName = gaulNames[state]
  roi = ee.FeatureCollection('FAO/GAUL/2015/level1') \
          .filter(ee.Filter.eq('ADM1_NAME', gaulName)).first() \
          .set('state', state)
  return roi

def clipToState(roi):
  img = distImgIndia.clip(roi.geometry())
  return img.set('state', roi.get('state'))

def getImgCollection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})['assets']
  image_list = [ee.Image(asset['name']) for asset in asset_list]
  return ee.ImageCollection(image_list)


# Export road rasters pan-India

In [ ]:
# Loop through states to collect road vectors
roadPolygons = ee.FeatureCollection([])
for state in states:
  print(f"Processing {state}")
  roads = ee.FeatureCollection(asset_base_path + state)
  roadPolygons = roadPolygons.merge(roads)

Processing andhra_pradesh
Processing arunachal_pradesh
Processing assam
Processing bihar
Processing chhattisgarh
Processing gujarat
Processing haryana
Processing himachal_pradesh
Processing jharkhand
Processing karnataka
Processing kerala
Processing madhya_pradesh
Processing maharashtra
Processing manipur
Processing meghalaya
Processing mizoram
Processing nagaland
Processing odisha
Processing punjab
Processing rajasthan
Processing sikkim
Processing tamil_nadu
Processing telangana
Processing tripura
Processing uttar_pradesh
Processing uttarakhand
Processing west_bengal
Processing jammu_and_kashmir
Processing ladakh


In [ ]:
count = 0
def list_assets(folder_path):
  global count
  asset_list = ee.data.listAssets({'parent': folder_path})
  for asset in asset_list['assets']:
      print(asset['name'])
      count += 1


list_assets(asset_base_path)
print(count)

projects/ee-corestackdev/assets/datasets/Road_DRRP/andhra_pradesh
projects/ee-corestackdev/assets/datasets/Road_DRRP/arunachal_pradesh
projects/ee-corestackdev/assets/datasets/Road_DRRP/assam
projects/ee-corestackdev/assets/datasets/Road_DRRP/bihar
projects/ee-corestackdev/assets/datasets/Road_DRRP/chhattisgarh
projects/ee-corestackdev/assets/datasets/Road_DRRP/gujarat
projects/ee-corestackdev/assets/datasets/Road_DRRP/haryana
projects/ee-corestackdev/assets/datasets/Road_DRRP/himachal_pradesh
projects/ee-corestackdev/assets/datasets/Road_DRRP/jammu_and_kashmir
projects/ee-corestackdev/assets/datasets/Road_DRRP/jharkhand
projects/ee-corestackdev/assets/datasets/Road_DRRP/karnataka
projects/ee-corestackdev/assets/datasets/Road_DRRP/kerala
projects/ee-corestackdev/assets/datasets/Road_DRRP/ladakh
projects/ee-corestackdev/assets/datasets/Road_DRRP/madhya_pradesh
projects/ee-corestackdev/assets/datasets/Road_DRRP/maharashtra
projects/ee-corestackdev/assets/datasets/Road_DRRP/manipur
projec

Get distance-from-roads image pan-India

In [ ]:
# Road mask
roadImgIndia = ee.Image().byte().paint(roadPolygons, 1)

distImgIndia = roadImgIndia.fastDistanceTransform() \
        .sqrt() \
        .multiply(ee.Image.pixelArea().sqrt()) \
        .reproject(crs='EPSG:4326', scale=scale) \
        .clip(indiaACZs.geometry())
print(distImgIndia.getInfo())

Output hidden; open in https://colab.research.google.com to view.

Get collection of state boundaries and clip distance-from-roads image to each boundary to create image collection

In [ ]:
stateRoiList = []
for state in states:
  if state == 'jammu_and_kashmir' or state == 'ladakh':
    continue
  stateRoiList.append(getStateRoi(state))

stateRois = ee.Feature(ee.FeatureCollection(stateRoiList).union().first())
indiaRoi = ee.Feature(indiaACZs.union().first())
diff = indiaRoi.difference(stateRois)
# distImgStates = ee.ImageCollection(stateRois.map(clipToState))


In [ ]:
print(diff.geometry().getInfo())

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
distImgDiff = distImgIndia.clip(diff.geometry())

In [ ]:
roads_img = getImgCollection('projects/ee-mtpictd/assets/anoushka/road_distances').mosaic().clip(indiaACZs.geometry())

In [ ]:
Map = geemap.Map()
visParams = {
    'min': 0,
    'max': 10000,
    'palette': ['ffffff', '006400', '003c00', '001e00']
}
# Map.addLayer(distImgIndia.clip(diff.geometry()), visParams, 'Road Distance')
Map.addLayer(stateRois, {}, 'State Boundaries')
# Map.addLayer(distImgDiff, visParams, 'Road Distance')
Map.addLayer(indiaRoi, {}, 'India Boundaries')
Map.addLayer(diff, {}, 'Diff')
Map.centerObject(indiaRoi, 4)
Map


Map(center=[23.221038363300835, 79.46098884927167], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
task = ee.batch.Export.image.toAsset(
    image=distImgDiff,
    description=f'RoadDistance_diff',
    assetId=f'projects/ee-mtpictd/assets/anoushka/road_distances/diff',
    region=diff.geometry(),
    scale=scale,
    crs='EPSG:4326',
    maxPixels=1e13
)
task.start()

# Export distance from water sources rasters pan-India

## Exporting

In [ ]:
def get_water_mask(aoi, year, scale = 10):
  year = int(year)
  indiasat_asset = f"projects/ee-corestackdev/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_{year}_{year+1}"
  lulc_image = ee.Image(indiasat_asset).select("predicted_label").clip(aoi)
  lulc_mask = lulc_image.eq(2).Or(lulc_image.eq(3)).Or(lulc_image.eq(4))
  return lulc_image.updateMask(lulc_mask).reproject(crs='EPSG:4326', scale=scale)


In [ ]:
roi = indiaACZs

water_img = get_water_mask(roi, year=2022, scale=100)
dist_img = water_img \
    .fastDistanceTransform() \
    .sqrt() \
    .multiply(ee.Image.pixelArea().sqrt()) \
    .reproject(crs='EPSG:4326', scale=scale) \
    .clip(roi.geometry())

In [ ]:
for state in states:
  stateRoi = getStateRoi(state)
  img = dist_img.clip(stateRoi.geometry())
  task = ee.batch.Export.image.toAsset(
      image=img,
      description=f'Water_Distance_{state}',
      assetId=f'projects/ee-mtpictd/assets/anoushka/dist_water_2022/{state}',
      region=stateRoi.geometry(),
      scale=100,
      crs='EPSG:4326',
      maxPixels=1e13
  )
  task.start()
  print(f"Exporting {state}")

Exporting andhra_pradesh
Exporting arunachal_pradesh
Exporting assam
Exporting bihar
Exporting chhattisgarh
Exporting gujarat
Exporting haryana
Exporting himachal_pradesh
Exporting jharkhand
Exporting karnataka
Exporting kerala
Exporting madhya_pradesh
Exporting maharashtra
Exporting manipur
Exporting meghalaya
Exporting mizoram
Exporting nagaland
Exporting odisha
Exporting punjab
Exporting rajasthan
Exporting sikkim
Exporting tamil_nadu
Exporting telangana
Exporting tripura
Exporting uttar_pradesh
Exporting uttarakhand
Exporting west_bengal


In [ ]:
stateRoiList = []
for state in states:
  stateRoiList.append(getStateRoi(state))

stateRois = ee.Feature(ee.FeatureCollection(stateRoiList).union().first())
aczRoi = ee.Feature(indiaACZs.union().first())
diff = aczRoi.difference(stateRois)
print(diff.getInfo())

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
state = "diff"
img = dist_img.clip(diff.geometry())
task = ee.batch.Export.image.toAsset(
    image=img,
    description=f'Water_Distance_{state}',
    assetId=f'projects/ee-mtpictd/assets/anoushka/dist_water_2022/{state}',
    region=diff.geometry(),
    scale=100,
    crs='EPSG:4326',
    maxPixels=1e13
)
task.start()
print(f"Exporting {state}")

Exporting diff


## Visualization

In [ ]:
dist_water_imgs = getImgCollection('projects/ee-mtpictd/assets/anoushka/dist_water_2022').mosaic().clip(indiaACZs.geometry())

In [ ]:
visparams = {
    'min': 0,
    'max': 10000,
    'palette': ['#08306B', '#2171B5', '#6BAED6', '#C6DBEF', '#F7FBFF']
}

In [ ]:
Map = geemap.Map()
# Map.addLayer(kolar_dist_img, visparams, 'Kolar')
# Map.addLayer(water_img.clip(roi.geometry()), {'palette': ['blue']}, 'India Water Mask')
# Map.addLayer(dist_img, visparams, 'India')
Map.addLayer(dist_water_imgs, visparams, 'India')
Map.centerObject(indiaACZs, 4)
Map

Map(center=[23.22103836473429, 79.4609888487962], controls=(WidgetControl(options=['position', 'transparent_bg…